# Figure 5 + Supplement (best 2022 analogue / 25 June 2026) — fixed 1995–2025 climatology

Same analogue selection as `ANALOGUES_timing_anomaly_from_catalogue_version3-5` (large classification box,
`ex==1` closed cut-offs from the `col_features_AMJJA` catalogue, Z500-distance ranking with ±5-day
independence), but **all T2M anomalies use the fixed climatology of `paper_plot` Fig. 1**:
ERA5 0.5° daily-mean T2M, **1995–2025**, mean over all days within **±3 days** of the calendar day
(7-day window). The present-climate (RMST) projection is applied on top, unchanged.
The RMST->T2M slope of that projection is fitted on all AMJJA days 1940–2025 (event year 2026 excluded).

* **Figure 5** — (a) analogue timing, (b) 5-day COL activity vs present-climate T2M anomaly
  (2026 heatwaves as black dots if within the top-85), (c) top-5 COL-activity T2M-anomaly composite
* **Numbers for the text** — Noyelle et al. (Robin) bootstrap attribution per pool (cut-off / no cut-off):
  RMST slope, 1950→2025 warming with 95 % CI, K–S pass rate; plus COL activity vs T2M correlations
  (raw, ridge-removed, partial) for both the Fig. 5 projection and the per-pool projection
* **Numbers behind Fig. 5b** — cut-off sub-pool stats: raw r, ridge-controlled partial r, pool-size
  dependence, ridge-controlled slope with bootstrap CI
* **Supplement** — panel (b) re-computed for two reference patterns: best 2022 analogue of the
  25 May event, and the 25 June 2026 heatwave

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import statsmodels.api as sm
import sklearn.metrics.pairwise as skmp
from scipy import stats

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import BoundaryNorm

mpl.rcParams["font.size"] = 9
mpl.rcParams["figure.dpi"] = 150
color2 = (180 / 256, 10 / 256, 10 / 256)          # factual red (matches the analogue notebooks)


import urllib.request

IPCC_TEMP_DIV_FILE = "temp_div.txt"
IPCC_TEMP_DIV_URLS = [
    "https://raw.githubusercontent.com/IPCC-WG1/colormaps/master/"
    "continuous_colormaps_rgb_0-1/temp_div.txt",
    "https://raw.githubusercontent.com/IPCC-WG1/colormaps/main/"
    "continuous_colormaps_rgb_0-1/temp_div.txt",
]
# Local locations to try before downloading (cwd, then Robin's Levante copy).
IPCC_TEMP_DIV_LOCAL = [
    IPCC_TEMP_DIV_FILE,
    "${HOME}/colormaps_IPCC/continuous_colormaps_rgb_0-1/temp_div.txt",
]


def load_temp_div(fallback="RdBu_r"):
    """IPCC temp_div: local file -> download from IPCC-WG1 -> Matplotlib fallback."""
    for p in IPCC_TEMP_DIV_LOCAL:
        if os.path.exists(p):
            return mcolors.LinearSegmentedColormap.from_list("temp_div", np.loadtxt(p))
    for url in IPCC_TEMP_DIV_URLS:
        try:
            urllib.request.urlretrieve(url, IPCC_TEMP_DIV_FILE)
            print("Downloaded IPCC temp_div from", url)
            return mcolors.LinearSegmentedColormap.from_list(
                "temp_div", np.loadtxt(IPCC_TEMP_DIV_FILE))
        except Exception as exc:
            print("  could not fetch", url, "->", exc)
    print("IPCC temp_div unavailable; falling back to", fallback)
    return plt.get_cmap(fallback)


cmap_temp_div = load_temp_div()


def squeeze_auxiliary_dimensions(data):
    for dim in list(data.dims):
        if dim not in ("time", "lat", "lon"):
            if data.sizes[dim] != 1:
                raise ValueError(f"Unexpected non-singleton dimension {dim}: {data.sizes[dim]}")
            data = data.isel({dim: 0}, drop=True)
    aux = [c for c in data.coords if c not in data.dims and c != "time"]
    return data.drop_vars(aux) if aux else data


def select_latitude_range(field, minimum, maximum):
    if bool(field["lat"][0] < field["lat"][-1]):
        return field.sel(lat=slice(minimum, maximum))
    return field.sel(lat=slice(maximum, minimum))


def comp_target_box_mean(data, lon_name, lat_name):
    """cos(lat)-weighted box mean, each cell also weighted by the fraction of its
    area inside the target box -> grid-independent (no half-degree edge jumps)."""
    lon, lat = data[lon_name], data[lat_name]
    dlon = float(abs(lon.values[1] - lon.values[0]))
    dlat = float(abs(lat.values[1] - lat.values[0]))
    fx = ((np.minimum(lon + dlon/2, COMP_BOX_LON_MAX) -
           np.maximum(lon - dlon/2, COMP_BOX_LON_MIN)) / dlon).clip(0, 1)   # lon overlap fraction
    fy = ((np.minimum(lat + dlat/2, COMP_BOX_LAT_MAX) -
           np.maximum(lat - dlat/2, COMP_BOX_LAT_MIN)) / dlat).clip(0, 1)   # lat overlap fraction
    w = fx * fy * np.cos(np.deg2rad(lat))                                   # 2D via broadcasting
    valid = np.isfinite(data)
    return (data.where(valid) * w).sum((lon_name, lat_name)) / (w * valid).sum((lon_name, lat_name))


def _fit_skewnorm(values):
    """Method-of-moments skew-normal fit (identical to the analogue notebooks)."""
    v = np.asarray(values, dtype=float)
    mean, std = v.mean(), v.std()
    skew = np.mean((v - mean) ** 3) / std ** 3
    sp = min(0.99, abs(skew))
    delta = (np.sqrt(np.pi / 2 * sp ** (2 / 3) / (sp ** (2 / 3) + ((4 - np.pi) / 2) ** (2 / 3)))
             * np.sign(skew))
    shape = delta / np.sqrt(1 - delta ** 2)
    scale = std / np.sqrt(1 - 2 * delta ** 2 / np.pi)
    loc = mean - scale * delta * np.sqrt(2 / np.pi)
    return shape, loc, scale


def fixed_doy_climatology(da, start_year, end_year, half_window):
    """Fixed calendar-day climatology, identical to paper_plot Fig. 1
    (era5_t2m_0p5_climatology): for every day-of-year D, the mean over ALL days in
    start_year..end_year whose day-of-year lies within +/-half_window of D
    (half_window=3 -> 7-day window), with the same circular doy distance
    |((doy - D + 182) % 365) - 182|. Computed for D = 1..366 in one pass
    (per-doy sums/counts, then window-summed), NaN-aware like xarray's mean.
    Returns a DataArray with dim 'dayofyear' (1..366) + the remaining dims of `da`."""
    t = pd.DatetimeIndex(da["time"].values)
    base = da.isel(time=np.where((t.year >= start_year) & (t.year <= end_year))[0])
    if base.sizes["time"] == 0:
        raise ValueError(f"No days in the climatology baseline {start_year}-{end_year}.")
    doy_all = np.arange(1, 367)
    grp = base["time.dayofyear"]
    sums = base.fillna(0.0).groupby(grp).sum("time").reindex(dayofyear=doy_all, fill_value=0.0)
    cnts = base.notnull().groupby(grp).sum("time").reindex(dayofyear=doy_all, fill_value=0)
    other = [d for d in sums.dims if d != "dayofyear"]
    sums = sums.transpose("dayofyear", *other); cnts = cnts.transpose("dayofyear", *other)
    dist = np.abs(((doy_all[None, :] - doy_all[:, None] + 182) % 365) - 182)   # [target, source]
    W = (dist <= half_window).astype(float)
    S = np.tensordot(W, sums.values.astype(float), axes=(1, 0))
    N = np.tensordot(W, cnts.values.astype(float), axes=(1, 0))
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(N > 0, S / N, np.nan)
    print(f"[clim] fixed {start_year}-{end_year}, +/-{half_window} d window "
          f"({base.sizes['time']} baseline days)")
    return xr.DataArray(clim, dims=("dayofyear", *other),
                        coords={"dayofyear": doy_all, **{d: sums[d] for d in other}})


def _lighten(color, factor=0.6):
    c = np.array(mcolors.to_rgb(color))
    return tuple(1.0 - factor * (1.0 - c))

# --- rasterize pcolormesh/pcolor in vector output (PDF/SVG) so maps match the PNG (no inter-cell seams) ---
import matplotlib.axes as _mpl_axes_raster, matplotlib.pyplot as _plt_raster
for _pcm_name in ("pcolormesh", "pcolor"):
    _pcm_orig = getattr(_mpl_axes_raster.Axes, _pcm_name)
    if not getattr(_pcm_orig, "_rasterized_patch", False):
        def _pcm_make(_orig):
            def _wrapped(self, *a, **k):
                k.setdefault("rasterized", True)
                _m = _orig(self, *a, **k)
                try: _m.set_edgecolor("face")
                except Exception: pass
                return _m
            _wrapped._rasterized_patch = True
            return _wrapped
        setattr(_mpl_axes_raster.Axes, _pcm_name, _pcm_make(_pcm_orig))
_plt_raster.rcParams["savefig.dpi"] = 200   # crisp rasterized meshes in the PDF (PNG keeps its explicit dpi)
# --- end rasterize patch ---


In [ ]:
# ============================ CONFIGURATION ============================
path_project = "${ANALOGUE_BASE}/"
path_ERA5    = os.path.join(path_project, "rawdat/")
path2_ERA5    = os.path.join(path_project, "rawdat_0p5/")

COL_Z500_FILE = os.path.join(path_ERA5, "era5_z500_daily_NA_1940_2026_FULL_1deg.nc")
#T2M_FILE      = os.path.join(path_ERA5, "era5_t2m_daily_NA_1940_2026_FULL_1deg.nc")
T2M_0P5_FILE  = os.path.join(path2_ERA5, "era5_t2m_daily_NA_1940_2026_FULL_0p5deg.nc")  # 0.5-deg daily mean (Fig 5 c/d)
RMST_FULL_FILE = T2M_0P5_FILE   # 0.5-deg t2m (consistent with panels b & c)
TXX_FILE = "${MSE_BASE}/TXX/txx_at_tasmax_merged_1940_2025.nc"

# Pre-computed cut-off/trough classification (one row per detected feature, large box).
CATALOGUE_NC = os.path.join(path_project, "col_catalogue_AMJJA",
                            "col_features_AMJJA_1940_2025_largebox.nc")

EVENT_DATE = "2026-05-25"
EVENT = pd.Timestamp(EVENT_DATE).normalize()
EVENT_TAG = EVENT.strftime("%Y-%m-%d")
BASE_CASE_NAME = "AMJJA"
LAST_COMPLETE_ERA5_YEAR = 2025
ANALOGUE_POOL_START = pd.Timestamp(1940, 1, 1)
ANALOGUE_POOL_END   = pd.Timestamp(2025, 12, 31)
EXCLUDE_EVENT_YEAR_FROM_ANALOGUES = EVENT.year > LAST_COMPLETE_ERA5_YEAR

# Analogue-distance box + season (identical to both notebooks).
BASE_BOX = {"min_lat": 30, "max_lat": 68, "min_lon": -20, "max_lon": 25,
            "min_month": 4, "max_month": 8, "variable_analogues": "z500"}
COL_INPUT_IS_GEOPOTENTIAL = True

# Large classification box + ex==1 (closed cut-off) threshold. Trough threshold = inf.
COL_LON_MIN, COL_LON_MAX = -35, 0
COL_LAT_MIN, COL_LAT_MAX = 25, 50
COL_CUTOFF_SO_MIN = 6.0
INDEPENDENCE_DAYS = 5                 # +/- day independence in analogue ranking

# Target region + present-climate anomaly settings.
TARGET_LON_MIN, TARGET_LON_MAX = -5.5, 5
TARGET_LAT_MIN, TARGET_LAT_MAX = 44.5, 55.0
COMP_BOX_LON_MIN, COMP_BOX_LON_MAX = TARGET_LON_MIN, TARGET_LON_MAX   # box of comp_target_box_mean (paper_plot Fig. 1)
COMP_BOX_LAT_MIN, COMP_BOX_LAT_MAX = TARGET_LAT_MIN, TARGET_LAT_MAX
# Fixed T2M climatology = paper_plot Fig. 1 definition: 1995-2025, +/-3-day (7-day) window.
CLIM_START_YEAR = 1995
CLIM_END_YEAR = 2025
CLIM_HALF_WINDOW_DAYS = 3
PRESENT_CLIMATE_DETREND = True
ANOMALY_THRESHOLDS_C = [2.0, 4.0, 6.0]
PROBABILITY_SIZES = [40, 50, 85, 100]
N_SELECT = max(PROBABILITY_SIZES)     # best-N independent analogues kept per class

# d4 / cut-off day window + TXx map settings.
D4_WIN = 4                            # d4 = 5-day rolling sum [D-4, D]
MAPBOX = dict(lon=(-30, 30), lat=(30, 65))

GROUPS = [("cut-off", "COL_TROUGH", True), ("no cut-off", "NO_COL_TROUGH", False)]
case_colors = {"COL_TROUGH": color2, "NO_COL_TROUGH": (112 / 256, 160 / 256, 205 / 256)}
info_event = {"day": EVENT.strftime("%d"), "month": EVENT.strftime("%m"), "year": EVENT.strftime("%Y")}
base_case = {**BASE_BOX, "name": BASE_CASE_NAME,
             "exclude_event_year": EXCLUDE_EVENT_YEAR_FROM_ANALOGUES,
             "pool_start": ANALOGUE_POOL_START, "pool_end": ANALOGUE_POOL_END}
print("Event:", EVENT_TAG, "| pool", ANALOGUE_POOL_START.date(), "to", ANALOGUE_POOL_END.date())

In [ ]:
# ================= RMST (FULL) + RMST-detrended Z500 candidate pool =================
def _to_dt64(v):
    return np.datetime64(pd.Timestamp(v).to_datetime64())


def _month_mask(field, lo, hi):
    if hi >= lo:
        return (field["time.month"] >= lo) & (field["time.month"] <= hi)
    return (field["time.month"] >= lo) | (field["time.month"] <= hi)


YEARS_ROLLING = 15        # centred rolling-mean length for RMST [years]
YEARS_EXTRAPOLATION = 30  # years of smoothed RMST used to fit the linear edge extrapolation
RMST_LAT = (35, 70)       # European RMST box (unchanged from the original notebook)
RMST_LON = (-15, 30)


def load_rmst_full():
    """Daily European RMST: cos(lat)-weighted box mean of daily T2M, 15-yr centred
    running mean; the NaN edges left by the window are filled with a linear trend
    fitted to the first / last 30 years of the smoothed series (continuous at the joins).
    Daily adaptation of the monthly get_RMST_europe."""
    src = xr.open_dataarray(RMST_FULL_FILE)
    try:
        da = squeeze_auxiliary_dimensions(src)
        da = da.assign_coords(lon=(da["lon"] + 180) % 360 - 180).sortby("lon")
        da = select_latitude_range(da.sel(lon=slice(*RMST_LON)), *RMST_LAT)
        print(f"[RMST] box lat {float(da.lat.min()):.1f}..{float(da.lat.max()):.1f}, "
              f"lon {float(da.lon.min()):.1f}..{float(da.lon.max()):.1f} "
              f"(requested {RMST_LAT}, {RMST_LON})")
        weights = np.cos(np.deg2rad(da["lat"]))
        rmst = da.weighted(weights).mean(dim=("lat", "lon")).load()
    finally:
        src.close()
    if float(rmst.mean()) > 150:
        rmst = rmst - 273.15
    rmst["time"] = rmst.indexes["time"].normalize()

    # 15-yr centred running mean (full window required -> NaN at both edges)
    rmst = rmst.rolling(time=int(round(YEARS_ROLLING * 365.25)), center=True).mean()
    valid = rmst.dropna("time")
    n_fit = int(round(YEARS_EXTRAPOLATION * 365.25))
    t_head = rmst["time"] < valid["time"][0]
    t_tail = rmst["time"] > valid["time"][-1]

    def _edge_trend(fit_part, anchor, times):
        coef = fit_part.polyfit(dim="time", deg=1).polyfit_coefficients
        line = xr.polyval(times, coef)
        return line - (float(xr.polyval(anchor["time"], coef)) - float(anchor))  # pass through anchor

    trend_past = _edge_trend(valid[:n_fit], valid[0], rmst["time"][t_head])
    trend_future = _edge_trend(valid[-n_fit:], valid[-1], rmst["time"][t_tail])
    print(f"[RMST] filled {int(t_head.sum())} leading / {int(t_tail.sum())} trailing days by extrapolation")
    return rmst.fillna(trend_past).fillna(trend_future).rename("rmst")


def _restrict_pool(fields, case):
    if case.get("exclude_event_year", False):
        fields = fields.where(fields["time.year"] < int(info_event["year"]), drop=True)
    if case.get("pool_start") is not None:
        fields = fields.where(fields["time"] >= _to_dt64(case["pool_start"]), drop=True)
    if case.get("pool_end") is not None:
        fields = fields.where(fields["time"] <= _to_dt64(case["pool_end"]), drop=True)
    return fields


def load_detrended_fields_and_event(case, rmst):
    with xr.open_dataarray(COL_Z500_FILE) as source:
        z500 = source.sel(lon=slice(case["min_lon"], case["max_lon"]))
        z500 = select_latitude_range(z500, case["min_lat"], case["max_lat"])
        z500 = z500.where(_month_mask(z500, case["min_month"], case["max_month"]), drop=True).load()
    z500 = squeeze_auxiliary_dimensions(z500).transpose("time", "lat", "lon")

    weights = np.cos(np.deg2rad(z500["lat"]))
    z500_trend = z500.weighted(weights).mean(dim=("lat", "lon"))
    rmst_trend = rmst.where(_month_mask(rmst, case["min_month"], case["max_month"]), drop=True)
    z500_trend, rmst_trend = xr.align(z500_trend, rmst_trend, join="inner")
    z500 = z500.sel(time=z500_trend["time"])
    trend = np.polyfit(rmst_trend.values, z500_trend.values, 1)
    mean_rmst = float(rmst_trend.mean().values)
    global Z500_DETREND_SLOPE, Z500_DETREND_MEAN_RMST          # expose for 2026-peak distance
    Z500_DETREND_SLOPE = float(trend[0]); Z500_DETREND_MEAN_RMST = mean_rmst
    z500 = z500 - xr.DataArray(data=(rmst_trend - mean_rmst) * trend[0],
                               coords=z500["time"].coords, dims=["time"])
    if COL_INPUT_IS_GEOPOTENTIAL:
        z500 = z500 / 9.81
    z500 = z500.rename("z500")
    z500["time"] = z500.indexes["time"].normalize()

    event_field = z500.sel(time=np.datetime64(EVENT.to_datetime64()))
    fields = _restrict_pool(z500, case)
    return fields, event_field


RMST = load_rmst_full()
fields, event_field = load_detrended_fields_and_event(base_case, RMST)
print("Candidate pool:", dict(fields.sizes), "| event field:", dict(event_field.sizes))


In [ ]:
# ============ Classify the pool from the COL catalogue (ex==1, So>=6) ============
# A day is FACTUAL (COL_TROUGH) iff it carries >=1 qualifying closed cut-off in the
# large box; the catalogue is already restricted to that box, so we only threshold.
cat = xr.open_dataset(CATALOGUE_NC)
cdf = cat[["time", "So", "ex"]].to_dataframe().reset_index()
cat.close()

cc = cdf[(cdf["ex"] == 1) & (cdf["So"] >= COL_CUTOFF_SO_MIN)].copy()
cc["day"] = pd.DatetimeIndex(cc["time"]).normalize()
daily = cc.groupby("day")["So"].sum()
daily = daily.reindex(pd.date_range(daily.index.min(), daily.index.max(), freq="D"), fill_value=0.0)
d4_series   = daily.rolling(D4_WIN + 1, min_periods=1).sum()     # continuous d4 index
cutoff_days = daily[daily > 0].index                            # days with a qualifying cut-off
cutoff_set  = set(pd.DatetimeIndex(cutoff_days).normalize())

# Rank the pool by Z500 distance to the event; boolean class per pool day.
pool_times  = pd.DatetimeIndex(fields["time"].values).normalize()
pool_values = np.asarray(fields.values, dtype=float)
event_value = np.asarray(event_field.values, dtype=float)
_diff = pool_values - event_value
dist_to_event = np.sqrt((_diff * _diff).reshape(pool_values.shape[0], -1).sum(axis=1))
pool_has_feature = np.array([d in cutoff_set for d in pool_times], dtype=bool)

for _l, _g, _f in GROUPS:
    print(f"{_l}: {int((pool_has_feature == _f).sum())} candidate days in the pool")


def greedy_independent(dist, times, max_n=None):
    """Globally +/-INDEPENDENCE_DAYS-independent ranking by ascending distance
    (streamlined rank_independent_analogue_candidates)."""
    dist = np.asarray(dist, dtype=float)
    avail = np.isfinite(dist)
    tvals = np.asarray(times.values)
    order = []
    limit = len(dist) if max_n is None else max_n
    while avail.any() and len(order) < limit:
        cand = np.flatnonzero(avail)
        chosen = cand[np.argmin(dist[cand])]
        order.append(chosen)
        sep = np.abs((tvals - tvals[chosen]).astype("timedelta64[D]").astype(float))
        avail &= sep > INDEPENDENCE_DAYS
    return np.asarray(order, dtype=int)


ranked = greedy_independent(dist_to_event, pool_times)
fac_order = ranked[pool_has_feature[ranked]]
cf_order  = ranked[~pool_has_feature[ranked]]
for _g, _o in [("COL_TROUGH", fac_order), ("NO_COL_TROUGH", cf_order)]:
    if len(_o) < N_SELECT:
        raise RuntimeError(f"{_g}: only {len(_o)} independent analogues, need {N_SELECT}.")
case_dates_full = {"COL_TROUGH": pool_times[fac_order[:N_SELECT]],
                   "NO_COL_TROUGH": pool_times[cf_order[:N_SELECT]]}
print("Selected", N_SELECT, "independent analogues per class.")


In [ ]:
# ============ Target-region T2M anomaly (present-climate projection) ============
with xr.open_dataarray(T2M_0P5_FILE) as _src:
    _t2m = squeeze_auxiliary_dimensions(_src).sel(lon=slice(TARGET_LON_MIN - 1, TARGET_LON_MAX + 1))  # 1-deg margin:
    _t2m = select_latitude_range(_t2m, TARGET_LAT_MIN - 1, TARGET_LAT_MAX + 1).load()              # all overlapping cells
if float(_t2m.mean()) > 150:
    _t2m = _t2m - 273.15
target_box_series = comp_target_box_mean(_t2m, "lon", "lat")   # area-fraction x cos(lat) weighted, as paper_plot Fig. 1
target_box_series["time"] = target_box_series.indexes["time"].normalize()

# Fixed climatology (paper_plot Fig. 1): 1995-2025, +/-3-day window around each calendar day.
target_clim_doy = fixed_doy_climatology(target_box_series, CLIM_START_YEAR, CLIM_END_YEAR,
                                        CLIM_HALF_WINDOW_DAYS)
_series_times = pd.DatetimeIndex(target_box_series["time"].values).normalize()


def _raw_anomaly(dates):
    idx = pd.DatetimeIndex(pd.to_datetime(np.asarray(dates))).normalize()
    absolute = target_box_series.sel(time=idx.values).values
    reference = target_clim_doy.sel(dayofyear=xr.DataArray(idx.dayofyear.values, dims="t")).values
    return absolute - reference


_rmst_series = RMST.reindex(time=target_box_series["time"], method="nearest")
_rmst_lookup = pd.Series(np.asarray(_rmst_series.values, dtype=float), index=_series_times)
_season = (_series_times.month >= 5) & (_series_times.month <= 8)   # MJJA projection fit (standardized; was AMJJA via BASE_BOX)
_all_anom = _raw_anomaly(target_box_series["time"].values)
_fit_days = _season & (_series_times.year <= LAST_COMPLETE_ERA5_YEAR)   # slope fit: MJJA 1940-2025 (no 2026)
_x, _y = _rmst_lookup.values[_fit_days], _all_anom[_fit_days]
_ok = np.isfinite(_x) & np.isfinite(_y)
RMST_ANOMALY_SLOPE = float(np.polyfit(_x[_ok], _y[_ok], 1)[0])

_event_year_used = min(EVENT.year, LAST_COMPLETE_ERA5_YEAR)
_event_ref = pd.Timestamp(year=_event_year_used, month=EVENT.month, day=EVENT.day).normalize()
RMST_EVENT = float(RMST.sel(time=np.datetime64(_event_ref), method="nearest"))
REF_1950 = pd.Timestamp(year=1950, month=EVENT.month, day=EVENT.day).normalize()
RMST_1950 = float(RMST.sel(time=np.datetime64(REF_1950), method="nearest"))


def target_region_anomaly(dates):
    idx = pd.DatetimeIndex(pd.to_datetime(np.asarray(dates))).normalize()
    anom = _raw_anomaly(idx)
    if PRESENT_CLIMATE_DETREND:
        anom = anom + RMST_ANOMALY_SLOPE * (RMST_EVENT - _rmst_lookup.reindex(idx).values)
    return anom


event_target_anomaly = float(target_region_anomaly([EVENT])[0])
print(f"Event target-region anomaly: {event_target_anomaly:+.2f} degC | "
      f"RMST present={RMST_EVENT:.3f}, 1950={RMST_1950:.3f}")

### Figure 5 — COL activity → surface temperature (1 row × 3 panels; 0.5° daily-mean T2M)

In [ ]:
# ============================== FIGURE 5 (1x3) : computation ==============================
# (a) analogue occurrence across seasons | (b) 5-day COL activity vs present-climate T2M anomaly
# (c) top-5 COL-activity T2M-anomaly composite over the SELECTED ANALOGUE DAYS (factual +
#     counterfactual), on the 0.5-deg daily-mean ERA5 T2M grid, deseasonalised.
from scipy import stats

N_AB, N_CD, NTOP = 85, 85, 25
N_MAP =85# N_SELECT                       # analogue days per class used for the composite
FS_L, FS_T, FS_LEG, FS_LET = 11, 10, 10, 11
MONTH_EDGES = np.array([91, 121, 152, 182, 213, 244])
MONTH_CENTERS = 0.5 * (MONTH_EDGES[:-1] + MONTH_EDGES[1:])
MONTH_LABELS = ["Apr", "May", "Jun", "Jul", "Aug"]


def _letter(ax, s, x=-0.12, y=1.05):
    ax.text(x, y, s, transform=ax.transAxes, fontweight="bold", fontsize=13, va="bottom", ha="right")


def _fit_stats(x, y):
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return None
    rp, pp = stats.pearsonr(x, y); rs, ps = stats.spearmanr(x, y)
    b, a = np.polyfit(x, y, 1)
    return rp, pp, rs, ps, b, a


# ---- COL-activity index d4 (rebuild at SO_MIN_D4) ----
SO_MIN_D4 = 6
D4_WIN    = 4
_cat = xr.open_dataset(CATALOGUE_NC)
_cdf = _cat[["time", "So", "ex"]].to_dataframe().reset_index(); _cat.close()
_cc = _cdf[(_cdf["ex"] == 1) & (_cdf["So"] >= SO_MIN_D4)].copy()
_cc["day"] = pd.DatetimeIndex(_cc["time"]).normalize()
_daily = _cc.groupby("day")["So"].sum()
_daily = _daily.reindex(pd.date_range(_daily.index.min(), _daily.index.max(), freq="D"), fill_value=0.0)
d4_series   = _daily.rolling(D4_WIN + 1, min_periods=1).sum()          # continuous 5-day COL activity
cutoff_days = _daily[_daily > 0].index
_m = _daily[(_daily.index.month >= BASE_BOX["min_month"]) & (_daily.index.month <= BASE_BOX["max_month"])]  # Apr-Aug
D4_NORM = float(_m[_m > 0].mean())   # mean daily Sum(So) on cut-off days, AMJJA 1940-2025 (~15 = NF of Figs 3/4)
print(f"So>={SO_MIN_D4}: cut-off days={len(cutoff_days)}  |  D4_NORM={D4_NORM:.2f} (AMJJA mean over cut-off days)")

# ---- SELECTED analogue days (factual + counterfactual), pooled ----
sel_days = pd.DatetimeIndex(np.concatenate([
    case_dates_full["COL_TROUGH"][:N_MAP].values,
    case_dates_full["NO_COL_TROUGH"][:N_MAP].values])).normalize().unique()
d4_sel = d4_series.reindex(sel_days).to_numpy()
print(f"selected analogue days for the composite: {len(sel_days)} (N_MAP={N_MAP} per class)")

# ---- 0.5-deg daily-mean T2M over MAPBOX; anomalies vs fixed 1995-2025 7-day-window climatology ----
with xr.open_dataarray(T2M_0P5_FILE) as _s2:
    _t2 = squeeze_auxiliary_dimensions(_s2).sel(lon=slice(*MAPBOX["lon"]))
    _t2 = select_latitude_range(_t2, MAPBOX["lat"][0], MAPBOX["lat"][1]).load()
if float(_t2.mean()) > 150:
    _t2 = _t2 - 273.15
_t2["time"] = _t2.indexes["time"].normalize()

# Fixed per-cell climatology (paper_plot Fig. 1): 1995-2025, +/-3-day window.
clim_doy = fixed_doy_climatology(_t2, CLIM_START_YEAR, CLIM_END_YEAR,
                                 CLIM_HALF_WINDOW_DAYS).transpose("dayofyear", "lat", "lon")

def _t2m_anom_days(days):
    idx = pd.DatetimeIndex(pd.to_datetime(np.asarray(days))).normalize()
    fld = _t2.sel(time=idx.values)
    ref = clim_doy.sel(dayofyear=xr.DataArray(idx.dayofyear.values, dims="time"))
    return fld.values - ref.values

A_sel = _t2m_anom_days(sel_days)
LATc = _t2["lat"].values.astype(float); LONc = _t2["lon"].values.astype(float)
_okd = np.all(np.isfinite(A_sel), axis=(1, 2)) & np.isfinite(d4_sel)
_xx = d4_sel[_okd]; _YY = A_sel[_okd]; _nok = int(_okd.sum())

# ---------- top-5 COL-activity T2M-anomaly composite (selected days) ----------
TOPN, B = 5, 10000
rng = np.random.default_rng(0)
_ord = np.argsort(_xx)[::-1][:TOPN]                                   # top-5 by d4 among selected days
_mean_all = _YY.mean(axis=0)
obs = _YY[_ord].mean(axis=0) - _mean_all                             # composite anomaly (degC)
_cnt = np.zeros(_YY.shape[1:], dtype=np.int64)                        # one-sided bootstrap
for _b in range(B):
    _pick = rng.choice(_nok, TOPN, replace=False)
    _cnt += (_YY[_pick].mean(axis=0) - _mean_all >= obs)
PMAP_arr = (_cnt + 1) / (B + 1)
ZMAP = xr.DataArray(obs, coords={"lat": LATc, "lon": LONc}, dims=("lat", "lon")).to_pandas()
PMAP = xr.DataArray(PMAP_arr, coords={"lat": LATc, "lon": LONc}, dims=("lat", "lon")).to_pandas()
mlon = LONc; mlat = LATc
print(f"composite over {_nok} days | sig@5%: {(PMAP_arr<=0.05).sum()} / {PMAP_arr.size} cells")

In [ ]:
# ============================================================================
# COL activity (d4) of each 2026 heatwave -- colindex2 on the 2026 z500, same
# detector as the catalogue. Plus each peak's detrended-Z500 distance to the
# event and the top-85 analogue distance threshold, so panel (b) keeps only the
# 2026 peaks that rank within the best-85 analogues.   (Levante + colindex2.)
# ============================================================================
import glob, os
from pathlib import Path
from colindex2 import Detect

Z500_FILE = COL_Z500_FILE                          # 1-deg z500, 1940-2026 FULL (incl. 2026)
DET_LON, DET_LAT = (-50, 55), (5, 90)
RADII = np.arange(200, 2101, 100)
SO_DETECT, SO_CUTOFF, SR_MAX = 3.0, 6.0, 3.0     # detector floor / study cut-off qualifier / SR cap
SAVE_SO_MIN = SO_DETECT                          # catalogue read floor (= COL_DETECTION_SO_MIN in the builder)

hw2026 = [pd.Timestamp("2026-05-25"), pd.Timestamp("2026-06-25"), pd.Timestamp("2026-07-10")]


def _load_z(dates):
    src = xr.open_dataarray(Z500_FILE); h = src.sel(time=pd.DatetimeIndex(dates)).load(); src.close()
    for d in list(h.dims):
        if d not in ("time", "lat", "lon"):
            h = h.isel({d: 0}, drop=True)
    h = h.sel(lon=slice(*DET_LON))
    h = h.sel(lat=slice(DET_LAT[1], DET_LAT[0])) if float(h.lat[0]) > float(h.lat[-1]) else h.sel(lat=slice(*DET_LAT))
    return (h / 9.80665).assign_attrs(units="m", standard_name="geopotential_height").assign_coords(level=500).transpose("time", "lat", "lon")


def _read_features_day(out_dir, ts):
    """Cut-off/trough features INSIDE the large box for one date -- byte-for-byte the same
    read + spatial constraint as the catalogue builder's read_features()."""
    ts = pd.Timestamp(ts)
    csv = Path(out_dir) / "V" / ts.strftime("%Y%m") / ("V-L-" + ts.strftime("%Y%m%d%H%M") + "-0500.csv")
    if not csv.exists():
        return pd.DataFrame(columns=["So", "ex"])
    pts = pd.read_csv(csv)
    if pts.empty:
        return pd.DataFrame(columns=["So", "ex"])
    lon180 = (pts["lon"] + 180) % 360 - 180
    keep = (lon180.between(COL_LON_MIN, COL_LON_MAX)          # <-- LARGE box constrains the selection
            & pts["lat"].between(COL_LAT_MIN, COL_LAT_MAX)    #     (identical to the catalogue)
            & pts["So"].ge(SAVE_SO_MIN))
    p = pts.loc[keep]
    return pd.DataFrame({"So": p["So"].values,
                         "ex": pd.to_numeric(p["ex"], errors="coerce").fillna(0).astype(int).values})


def detect_d4(peak):
    """d4 = 5-day sum over [peak-4, peak] of the daily Sum(So) of closed cut-offs
    (ex==1, So>=SO_CUTOFF) inside the large box -- built the SAME way as the catalogue
    that feeds d4_series (plain per-day Sum(So); colindex2 writes one row per feature)."""
    win = pd.date_range(pd.Timestamp(peak) - pd.Timedelta(days=4), peak, freq="D")
    outdir = Path(f"./colindex_hw_{pd.Timestamp(peak):%Y%m%d}"); outdir.mkdir(parents=True, exist_ok=True)
    prev = os.getcwd(); os.chdir(outdir)
    try:
        Detect(_load_z(win), odir="out", ty="L", r=RADII, SR_thres=SR_MAX, So_thres=SO_DETECT,
               Do_thres=0.0, xx_thres=0.0, rm_rmin=True, rm_rmax=True, local_ex_req_num=7,
               gpu=False, chunk_size=100_000, nc=True, subs=False)
    finally:
        os.chdir(prev)
    total = 0.0
    for d in win:                                            # daily Sum(So) of cut-offs, then 5-day sum
        f = _read_features_day(outdir / "out", d)
        total += float(f.loc[(f["ex"] == 1) & (f["So"] >= SO_CUTOFF), "So"].sum())
    return total


# --- detrended-Z500 distance of a date to the event (same metric as the analogue ranking) ---
event_value = np.asarray(event_field.values, dtype=float)


def _detrended_z500_field(date):
    with xr.open_dataarray(COL_Z500_FILE) as src:
        z = src.sel(lon=slice(BASE_BOX["min_lon"], BASE_BOX["max_lon"]))
        z = select_latitude_range(z, BASE_BOX["min_lat"], BASE_BOX["max_lat"])
        z = z.sel(time=np.datetime64(pd.Timestamp(date).normalize())).load()
    z = squeeze_auxiliary_dimensions(z)
    rmst_t = float(RMST.sel(time=np.datetime64(pd.Timestamp(date).normalize()), method="nearest"))
    z = z - (rmst_t - Z500_DETREND_MEAN_RMST) * Z500_DETREND_SLOPE
    if COL_INPUT_IS_GEOPOTENTIAL:
        z = z / 9.81
    return np.asarray(z.values, dtype=float)


def event_distance(date):
    return float(np.sqrt(np.sum((_detrended_z500_field(date) - event_value) ** 2)))


DIST_TOP85 = float(dist_to_event[ranked[:N_AB]].max())   # 85th-closest independent analogue
print(f"top-{N_AB} analogue distance threshold = {DIST_TOP85:.2f}")

# --- consistency check: detect_d4 must match the catalogue-based d4_series on a known 2025 day ---
_chk = pd.Timestamp("2025-06-17")                        # an active analogue day present in the catalogue
_cat_d4 = float(d4_series.reindex([_chk]).iloc[0])
_det_d4 = detect_d4(_chk)
print(f"[check] {_chk.date()}: detect_d4={_det_d4:.2f}  vs  catalogue d4_series={_cat_d4:.2f}  "
      f"(ratio {_det_d4/_cat_d4:.2f} — should be ~1)")

D4_2026, DIST_2026 = {}, {}
for pk in hw2026:
    key = str(pk.date())
    D4_2026[key] = detect_d4(pk)
    DIST_2026[key] = event_distance(pk)
    keep = DIST_2026[key] <= DIST_TOP85
    print(f"2026 {key}: d4={D4_2026[key]:6.2f} ({D4_2026[key]/D4_NORM:.2f}x clim)  "
          f"dist={DIST_2026[key]:7.2f}  -> {'KEEP (in top-85)' if keep else 'omit (outside top-85)'}")

In [ ]:
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MultipleLocator

# ---- font scheme, comparable to Figs 2–4 (single base fs, offsets not hardcoded) ----
fs    = 14
axfs  = fs-2   # x/y tick label size
legfs = fs-4   # in-panel legends / stats boxes
plt.rcParams.update({"font.size":fs,"axes.titlesize":fs+1,"axes.labelsize":fs,
                     "xtick.labelsize":axfs,"ytick.labelsize":axfs})
FS_L   = fs      # axis labels
FS_T   = axfs    # small marginal-axis labels ("count", "PDF")
FS_LEG = legfs   # in-panel legends

proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(16, 4))
outer = fig.add_gridspec(1, 3, width_ratios=[1., 1.35, 1.25], wspace=0.2)

def _letter(ax, s, x=-0.12, dy=6):   # dy in points: same gap above the top edge for every panel
    ax.annotate(s, xy=(x, 1.0), xycoords="axes fraction",
                xytext=(0, dy), textcoords="offset points",
                fontweight="bold", fontsize=fs+1, va="bottom", ha="right")

# ===== (a) analogue occurrence across seasons (timing scatter + marginals) =====
gsa = outer[0, 0].subgridspec(2, 2, width_ratios=[4, 1.2], height_ratios=[1, 4], wspace=0.02, hspace=0.04)
ax_top = fig.add_subplot(gsa[0, 0]); ax_main = fig.add_subplot(gsa[1, 0], sharex=ax_top)
ax_right = fig.add_subplot(gsa[1, 1], sharey=ax_main)
case_dates = {g: case_dates_full[g][:N_AB] for _, g, _ in GROUPS}
_yr_min = min(int(case_dates[g].year.min()) for _, g, _ in GROUPS)
_yr_max = max(int(case_dates[g].year.max()) for _, g, _ in GROUPS)
decade_bins = np.arange((_yr_min // 10) * 10, (_yr_max // 10) * 10 + 11, 10)
for label, group, _f in GROUPS:
    d = case_dates[group]; col = case_colors[group]
    ax_main.scatter(d.year.values.astype(float), d.dayofyear.values.astype(float),
                    s=15, color=col, alpha=0.7, edgecolors="none", label=label)
    ax_top.hist(d.year.values.astype(float), bins=decade_bins, histtype="stepfilled", alpha=0.35, color=col)
    ax_right.hist(d.dayofyear.values.astype(float), bins=MONTH_EDGES, histtype="stepfilled",
                  alpha=0.35, color=col, orientation="horizontal")
ax_main.axhline(EVENT.dayofyear, color="black", ls="--", lw=0.8)
ax_main.set_xlabel("Year", fontsize=FS_L); ax_main.set_ylabel("Month", fontsize=FS_L)
ax_main.set_yticks(MONTH_CENTERS); ax_main.set_yticklabels(MONTH_LABELS)
ax_main.set_ylim(MONTH_EDGES[0] - 2, MONTH_EDGES[-1] + 2)
ax_main.grid(linestyle="dotted", linewidth=0.3); ax_main.legend(fontsize=FS_LEG, frameon=False, loc="lower left")
ax_top.tick_params(labelbottom=False); ax_top.set_ylabel("count", fontsize=FS_T)
ax_right.tick_params(labelleft=False); ax_right.set_xlabel("count", fontsize=FS_T)
ax_top.grid(linestyle="dotted", linewidth=0.3); ax_right.grid(linestyle="dotted", linewidth=0.3)
_letter(ax_top, "(a)", x=-0.18)

# ===== (b) 5-day COL activity vs present-climate T2M anomaly + PDF marginal =====
gsb = outer[0, 1].subgridspec(1, 2, width_ratios=[4, 1.25], wspace=0.03)
ax_bs = fig.add_subplot(gsb[0, 0]); ax_bm = fig.add_subplot(gsb[0, 1], sharey=ax_bs)
anoms_by_group = {}
for label, group, _flag in GROUPS:
    dates = case_dates_full[group][:N_CD]
    d4_all = d4_series.reindex(dates).to_numpy() / D4_NORM          # normalized d4
    t2_all = target_region_anomaly(dates)                          # present climate (0.5-deg t2m)
    rank = np.arange(len(dates)); ok = np.isfinite(d4_all) & np.isfinite(t2_all)
    d4, t2, rank = d4_all[ok], t2_all[ok], rank[ok]
    top = rank < NTOP; col = case_colors[group]
    anoms_by_group[group] = t2
    ax_bs.scatter(d4[~top], t2[~top], s=14, color=col, alpha=0.22, edgecolors="none")
    ax_bs.scatter(d4[top], t2[top], s=34, color=col, alpha=0.9, edgecolors="k", linewidths=0.5)
    full = _fit_stats(d4, t2)
    if full:
        rp, pp, rs, ps, b, a = full; xs = np.linspace(d4.min(), d4.max(), 100)
        ax_bs.plot(xs, a + b * xs, color=col, lw=2.0,
                   label=f"{label} N={N_CD}: r={rp:+.2f} (p={pp:.2g})")# ρ={rs:+.2f}")
    t20 = _fit_stats(d4[top], t2[top])
    if t20:
        rp2, pp2, rs2, ps2, b2, a2 = t20; xs2 = np.linspace(d4[top].min(), d4[top].max(), 100)
        ax_bs.plot(xs2, a2 + b2 * xs2, color=col, lw=1.6, ls="--",
                   label=f"{label}  N={NTOP}: r={rp2:+.2f} (p={pp2:.2g})")#, ρ={rs2:+.2f}")
# ---- 2026 heatwaves as black dots, filtered to the top-85 analogues (detect_d4 COL activity) ----
for pk in hw2026:
    dist = event_distance(pk) if "event_distance" in globals() else 0.0
    if "DIST_TOP85" in globals() and dist > DIST_TOP85:
        print(f"[omit] 2026 {pk.date()} dist={dist:.2f} > top-{N_AB} threshold {DIST_TOP85:.2f}")
        continue
    d4v = D4_2026.get(str(pk.date())) if "D4_2026" in globals() else None
    if d4v is None and "detect_d4" in globals():
        d4v = detect_d4(pk)
    if d4v is None:
        print(f"[skip] no d4 for {pk.date()}"); continue
    x = d4v / D4_NORM; yv = float(target_region_anomaly([pk])[0])
    ax_bs.scatter(x, yv, s=44, color="black", zorder=9)
    ax_bs.annotate(pk.strftime("%m-%d"), (x, yv), textcoords="offset points",
                   xytext=(6, 3), fontsize=legfs-1, fontweight="bold", zorder=10)
    print(f"2026 {pk.strftime('%m-%d')}: d4={d4v:.1f}  x={x:.2f}  T2m anomaly={yv:+.2f} °C")
ax_bs.set_xlabel(r"COL Activity [Clim. Mean]", fontsize=FS_L)
ax_bs.set_ylabel("Box T2M anom., present [°C]", fontsize=FS_L)
ax_bs.grid(alpha=0.3); ax_bs.legend(fontsize=FS_LEG, frameon=True, loc="lower right")
_letter(ax_bs, "(b)", x=-0.18)

_allb = np.concatenate([anoms_by_group[g] for _, g, _ in GROUPS])
ygrid = np.linspace(_allb.min() - 1.0, _allb.max() + 1.0, 300)
for label, group, _f in GROUPS:
    col = case_colors[group]; ap = anoms_by_group[group]
    ax_bm.hist(ap, bins=18, density=True, orientation="horizontal", histtype="stepfilled", alpha=0.18, color=col)
    sh, lo, sc = _fit_skewnorm(ap)
    ax_bm.plot(stats.skewnorm.pdf(ygrid, sh, loc=lo, scale=sc), ygrid, color=col, lw=1.6)
    ax_bm.axhline(float(np.mean(ap)), color=col, ls="--", lw=0.9)
ax_bm.axhline(event_target_anomaly, color="black", lw=1.1)
for thr in ANOMALY_THRESHOLDS_C:
    ax_bm.axhline(thr, color="grey", ls=":", lw=0.6)
ax_bm.tick_params(labelleft=False, labelsize=FS_T); ax_bm.set_xlabel("PDF", fontsize=FS_T)
from matplotlib.ticker import FuncFormatter
ax_bm.xaxis.set_major_formatter(FuncFormatter(lambda v, _p: "" if abs(v) < 1e-9 else f"{v:g}"))
ax_bm.grid(linestyle="dotted", linewidth=0.3)

# ===== (c) top-5 COL-activity T2M-anomaly composite -- ENTIRE field, grey dots = insignificant =====
ax_c = fig.add_subplot(outer[0, 2], projection=proj)
dvmax, dstep = 6.0, 1.0
levels = np.round(np.arange(-dvmax, dvmax + dstep / 2, dstep), 2)
dnorm = BoundaryNorm(levels, ncolors=cmap_temp_div.N, clip=False)
ax_c.set_extent([-12, 10, 40, 60], crs=proj)
pmd = ax_c.pcolormesh(mlon, mlat, ZMAP.values, cmap=cmap_temp_div, norm=dnorm,
                      shading="auto", transform=proj)                       # entire field (unmasked)
_LON2, _LAT2 = np.meshgrid(mlon, mlat)
_insig = (PMAP.values > 0.05) & np.isfinite(ZMAP.values)                    # small grey dots on insignificant cells
ax_c.scatter(_LON2[_insig], _LAT2[_insig], s=15, c="grey", marker=".", linewidths=0,
             transform=proj, zorder=5)
_bx = [TARGET_LON_MIN, TARGET_LON_MAX, TARGET_LON_MAX, TARGET_LON_MIN, TARGET_LON_MIN]
_by = [TARGET_LAT_MIN, TARGET_LAT_MIN, TARGET_LAT_MAX, TARGET_LAT_MAX, TARGET_LAT_MIN]
ax_c.plot(_bx, _by, "k--", lw=1.0, transform=proj, zorder=7)
ax_c.add_feature(cfeature.COASTLINE, lw=1); ax_c.add_feature(cfeature.BORDERS, lw=0.3)
gld = ax_c.gridlines(draw_labels=True, ls="--", alpha=0.3); gld.top_labels = gld.right_labels = False
gld.xlocator = MultipleLocator(5); gld.ylocator = MultipleLocator(5)        # round coords only (no 44.5 etc.)
gld.xlabel_style = gld.ylabel_style = {"size": axfs}
ax_c.set_title("Top-5 COL-activity composite", fontsize=fs)
cbd = fig.colorbar(pmd, ax=ax_c, orientation="vertical", shrink=0.85, pad=0.02, ticks=levels)
cbd.set_label("T2M anomaly [°C] (dot: p > 0.05)", fontsize=fs-2)
cbd.ax.tick_params(labelsize=axfs)
_letter(ax_c, "(c)", x=-0.06)

plt.savefig("figure5_clim1995-2025.pdf", bbox_inches="tight")
plt.show()

### Figure 5 — vertical layout (panels stacked: a / b / c)

In [ ]:
# ==================== FIGURE 5 -- vertical layout (3 rows x 1 column) ====================
# Same content and data as the 1x3 Figure 5 cell above; only the panel arrangement differs.
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import MultipleLocator

# ---- font scheme, comparable to Figs 2–4 (single base fs, offsets not hardcoded) ----
fs    = 14
axfs  = fs-2   # x/y tick label size
legfs = fs-4   # in-panel legends / stats boxes
plt.rcParams.update({"font.size":fs,"axes.titlesize":fs+1,"axes.labelsize":fs,
                     "xtick.labelsize":axfs,"ytick.labelsize":axfs})
FS_L   = fs      # axis labels
FS_T   = axfs    # small marginal-axis labels ("count", "PDF")
FS_LEG = legfs   # in-panel legends

proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(6, 15))
GAP_AB, GAP_BC = 0.31, 0.39   # vertical gap between panels a-b and b-c (in units of a panel height)
outer = fig.add_gridspec(5, 1, height_ratios=[1, GAP_AB, 1, GAP_BC, 1.15], hspace=0)   # rows 0/2/4 = panels a/b/c

def _letter(ax, s, x=None, dy=6):   # vertical layout: all letters at the same figure x, dy points above the panel top
    ax.annotate(s, xy=(0.02, 1.0), xycoords=("figure fraction", "axes fraction"),
                xytext=(0, dy), textcoords="offset points",
                fontweight="bold", fontsize=fs+1, va="bottom", ha="left")

# ===== (a) analogue occurrence across seasons (timing scatter + marginals) =====
gsa = outer[0, 0].subgridspec(2, 2, width_ratios=[4, 1.2], height_ratios=[1, 4], wspace=0.04, hspace=0.04)
ax_top = fig.add_subplot(gsa[0, 0]); ax_main = fig.add_subplot(gsa[1, 0], sharex=ax_top)
ax_right = fig.add_subplot(gsa[1, 1], sharey=ax_main)
case_dates = {g: case_dates_full[g][:N_AB] for _, g, _ in GROUPS}
_yr_min = min(int(case_dates[g].year.min()) for _, g, _ in GROUPS)
_yr_max = max(int(case_dates[g].year.max()) for _, g, _ in GROUPS)
decade_bins = np.arange((_yr_min // 10) * 10, (_yr_max // 10) * 10 + 11, 10)
for label, group, _f in GROUPS:
    d = case_dates[group]; col = case_colors[group]
    ax_main.scatter(d.year.values.astype(float), d.dayofyear.values.astype(float),
                    s=15, color=col, alpha=0.7, edgecolors="none", label=label)
    ax_top.hist(d.year.values.astype(float), bins=decade_bins, histtype="stepfilled", alpha=0.35, color=col)
    ax_right.hist(d.dayofyear.values.astype(float), bins=MONTH_EDGES, histtype="stepfilled",
                  alpha=0.35, color=col, orientation="horizontal")
ax_main.axhline(EVENT.dayofyear, color="black", ls="--", lw=0.8)
ax_main.set_xlabel("Year", fontsize=FS_L); ax_main.set_ylabel("Month", fontsize=FS_L)
ax_main.set_yticks(MONTH_CENTERS); ax_main.set_yticklabels(MONTH_LABELS)
ax_main.set_ylim(MONTH_EDGES[0] - 2, MONTH_EDGES[-1] + 2)
ax_main.grid(linestyle="dotted", linewidth=0.3); ax_main.legend(fontsize=FS_LEG, frameon=False, loc="lower left")
ax_top.tick_params(labelbottom=False); ax_top.set_ylabel("count", fontsize=FS_T)
ax_right.tick_params(labelleft=False); ax_right.set_xlabel("count", fontsize=FS_T)
ax_top.grid(linestyle="dotted", linewidth=0.3); ax_right.grid(linestyle="dotted", linewidth=0.3)
_letter(ax_top, "(a)", x=-0.18)

# ===== (b) 5-day COL activity vs present-climate T2M anomaly + PDF marginal =====
gsb = outer[2, 0].subgridspec(1, 2, width_ratios=[4, 1.25], wspace=0.03)
ax_bs = fig.add_subplot(gsb[0, 0]); ax_bm = fig.add_subplot(gsb[0, 1], sharey=ax_bs)
anoms_by_group = {}
for label, group, _flag in GROUPS:
    dates = case_dates_full[group][:N_CD]
    d4_all = d4_series.reindex(dates).to_numpy() / D4_NORM          # normalized d4
    t2_all = target_region_anomaly(dates)                          # present climate (0.5-deg t2m)
    rank = np.arange(len(dates)); ok = np.isfinite(d4_all) & np.isfinite(t2_all)
    d4, t2, rank = d4_all[ok], t2_all[ok], rank[ok]
    top = rank < NTOP; col = case_colors[group]
    anoms_by_group[group] = t2
    ax_bs.scatter(d4[~top], t2[~top], s=14, color=col, alpha=0.22, edgecolors="none")
    ax_bs.scatter(d4[top], t2[top], s=34, color=col, alpha=0.9, edgecolors="k", linewidths=0.5)
    full = _fit_stats(d4, t2)
    if full:
        rp, pp, rs, ps, b, a = full; xs = np.linspace(d4.min(), d4.max(), 100)
        ax_bs.plot(xs, a + b * xs, color=col, lw=2.0,
                   label=f"{label} N={N_CD}: r={rp:+.2f} (p={pp:.2g})")# ρ={rs:+.2f}")
    t20 = _fit_stats(d4[top], t2[top])
    if t20:
        rp2, pp2, rs2, ps2, b2, a2 = t20; xs2 = np.linspace(d4[top].min(), d4[top].max(), 100)
        ax_bs.plot(xs2, a2 + b2 * xs2, color=col, lw=1.6, ls="--",
                   label=f"{label}  N={NTOP}: r={rp2:+.2f} (p={pp2:.2g})")#, ρ={rs2:+.2f}")
# ---- 2026 heatwaves as black dots, filtered to the top-85 analogues (detect_d4 COL activity) ----
for pk in hw2026:
    dist = event_distance(pk) if "event_distance" in globals() else 0.0
    if "DIST_TOP85" in globals() and dist > DIST_TOP85:
        print(f"[omit] 2026 {pk.date()} dist={dist:.2f} > top-{N_AB} threshold {DIST_TOP85:.2f}")
        continue
    d4v = D4_2026.get(str(pk.date())) if "D4_2026" in globals() else None
    if d4v is None and "detect_d4" in globals():
        d4v = detect_d4(pk)
    if d4v is None:
        print(f"[skip] no d4 for {pk.date()}"); continue
    x = d4v / D4_NORM; yv = float(target_region_anomaly([pk])[0])
    ax_bs.scatter(x, yv, s=44, color="black", zorder=9)
    ax_bs.annotate(pk.strftime("%m-%d"), (x, yv), textcoords="offset points",
                   xytext=(6, 3), fontsize=legfs-1, fontweight="bold", zorder=10)
    print(f"2026 {pk.strftime('%m-%d')}: d4={d4v:.1f}  x={x:.2f}  T2m anomaly={yv:+.2f} °C")
ax_bs.set_xlabel(r"COL Activity [Clim. Mean]", fontsize=FS_L)
ax_bs.set_ylabel("Box T2M anom., present [°C]", fontsize=FS_L)
ax_bs.grid(alpha=0.3); ax_bs.legend(fontsize=FS_LEG, frameon=True, loc="lower right")
_letter(ax_bs, "(b)", x=-0.18)

_allb = np.concatenate([anoms_by_group[g] for _, g, _ in GROUPS])
ygrid = np.linspace(_allb.min() - 1.0, _allb.max() + 1.0, 300)
for label, group, _f in GROUPS:
    col = case_colors[group]; ap = anoms_by_group[group]
    ax_bm.hist(ap, bins=18, density=True, orientation="horizontal", histtype="stepfilled", alpha=0.18, color=col)
    sh, lo, sc = _fit_skewnorm(ap)
    ax_bm.plot(stats.skewnorm.pdf(ygrid, sh, loc=lo, scale=sc), ygrid, color=col, lw=1.6)
    ax_bm.axhline(float(np.mean(ap)), color=col, ls="--", lw=0.9)
ax_bm.axhline(event_target_anomaly, color="black", lw=1.1)
for thr in ANOMALY_THRESHOLDS_C:
    ax_bm.axhline(thr, color="grey", ls=":", lw=0.6)
ax_bm.tick_params(labelleft=False, labelsize=FS_T); ax_bm.set_xlabel("PDF", fontsize=FS_T)
from matplotlib.ticker import FuncFormatter
ax_bm.xaxis.set_major_formatter(FuncFormatter(lambda v, _p: "" if abs(v) < 1e-9 else f"{v:g}"))
ax_bm.grid(linestyle="dotted", linewidth=0.3)

# ===== (c) top-5 COL-activity T2M-anomaly composite -- ENTIRE field, grey dots = insignificant =====
ax_c = fig.add_subplot(outer[4, 0], projection=proj)
dvmax, dstep = 6.0, 1.0
levels = np.round(np.arange(-dvmax, dvmax + dstep / 2, dstep), 2)
dnorm = BoundaryNorm(levels, ncolors=cmap_temp_div.N, clip=False)
ax_c.set_extent([-12, 10, 40, 60], crs=proj)
pmd = ax_c.pcolormesh(mlon, mlat, ZMAP.values, cmap=cmap_temp_div, norm=dnorm,
                      shading="auto", transform=proj)                       # entire field (unmasked)
_LON2, _LAT2 = np.meshgrid(mlon, mlat)
_insig = (PMAP.values > 0.05) & np.isfinite(ZMAP.values)                    # small grey dots on insignificant cells
ax_c.scatter(_LON2[_insig], _LAT2[_insig], s=15, c="grey", marker=".", linewidths=0,
             transform=proj, zorder=5)
_bx = [TARGET_LON_MIN, TARGET_LON_MAX, TARGET_LON_MAX, TARGET_LON_MIN, TARGET_LON_MIN]
_by = [TARGET_LAT_MIN, TARGET_LAT_MIN, TARGET_LAT_MAX, TARGET_LAT_MAX, TARGET_LAT_MIN]
ax_c.plot(_bx, _by, "k--", lw=1.0, transform=proj, zorder=7)
ax_c.add_feature(cfeature.COASTLINE, lw=1); ax_c.add_feature(cfeature.BORDERS, lw=0.3)
gld = ax_c.gridlines(draw_labels=True, ls="--", alpha=0.3); gld.top_labels = gld.right_labels = False
gld.xlocator = MultipleLocator(5); gld.ylocator = MultipleLocator(5)        # round coords only (no 44.5 etc.)
gld.xlabel_style = gld.ylabel_style = {"size": axfs}
ax_c.set_title("Top-5 COL-activity composite", fontsize=fs)
cbd = fig.colorbar(pmd, ax=ax_c, orientation="vertical", shrink=0.85, pad=0.02, ticks=levels)
cbd.set_label("T2M anomaly [°C] (dot: p > 0.05)", fontsize=fs-2)
cbd.ax.tick_params(labelsize=axfs)
_letter(ax_c, "(c)", x=-0.06)

plt.savefig("figure5_vertical_clim1995-2025.pdf", bbox_inches="tight")
plt.show()

### Numbers for the text — per-pool attribution (Noyelle et al.) + ridge-controlled correlations

In [ ]:
# ============================================================================
# Numbers for the text (Figure 5b itself keeps the single all-AMJJA-days RMST slope).
#
# [A] Robin's run_attribution (analogues_attribution_proba_rev_response.ipynb), per pool
#     (cut-off / no cut-off), B_BOOT resamples of the N_CD analogues:
#     y = deseasonalised target-box T2M anomaly, OLS on RMST of the analogue day
#     (optionally + ridge and/or + COL activity as extra covariates in the same OLS);
#     MoM skew-normal fit to the residuals (own shape per pool), K-S pass rate
#     (sqrt(n)*max|F(r_(i)) - i/n| < 1.36), change 1950 -> 2025 = beta_RMST * dRMST
#     with annual-mean RMST, as in rev_response.
# [B] COL activity vs present-climate T2M of the best-N analogues, in both projection
#     frameworks (old = all-AMJJA-days slope, target_region_anomaly | Robin = the pool's
#     own bootstrap-median slope), raw and ridge-controlled.
#     Ridge = comp_target_box_mean of the RMST-detrended Z500 over the TARGET box (as in the
#     cell-12 numbers). Ridge-removed T2M uses the pool's own ridge->T2M slope (fit on N_CD);
#     partial r residualises COL activity and T2M on the ridge within each best-N subset.
# ============================================================================
B_BOOT = 1000
ATTR_YEAR_PAST, ATTR_YEAR_PRESENT = 1950, LAST_COMPLETE_ERA5_YEAR   # 1950 -> 2025 (last full year, as Fig 5)
N_LIST_REPORT = [20, 25, 30, 40, 50, N_CD]
_rmst_annual = RMST.groupby("time.year").mean()
RMST_PAST    = float(_rmst_annual.sel(year=ATTR_YEAR_PAST))
RMST_PRESENT = float(_rmst_annual.sel(year=ATTR_YEAR_PRESENT))
DRMST = RMST_PRESENT - RMST_PAST

# ---- ridge index: target-box mean of the RMST-detrended Z500 (m), as a SEASONAL ANOMALY ----
# Remove the 1995-2025 +/-3 d doy climatology (analogue dates scatter across the season, so the
# ridge covariate must be the Z500 anomaly, not absolute Z500) -- consistent with cells 15 & 18.
_rep_tz = comp_target_box_mean(fields, "lon", "lat").load()
_rep_ridge = pd.Series(np.asarray(_rep_tz.values, float),
                       index=pd.DatetimeIndex(_rep_tz["time"].values).normalize())
_rep_rc = fixed_doy_climatology(_rep_tz, CLIM_START_YEAR, CLIM_END_YEAR, CLIM_HALF_WINDOW_DAYS)
_rep_ridge = _rep_ridge - _rep_rc.sel(dayofyear=xr.DataArray(_rep_ridge.index.dayofyear.values, dims="t")).values


def _ols_rs(Xs, Y):
    """Row-wise OLS over bootstrap resamples (Robin's fit_linear, extended to k covariates).
    Xs: (B, n, k), Y: (B, n) -> coef (B, k+1) [intercept first], residuals (B, n)."""
    A = np.concatenate([np.ones(Y.shape + (1,)), Xs], axis=2)
    coef = np.linalg.solve(np.einsum("bni,bnj->bij", A, A), np.einsum("bni,bn->bi", A, Y)[..., None])[..., 0]
    return coef, Y - np.einsum("bni,bi->bn", A, coef)


def _fit_skewnorm_rs(V):
    """Row-wise method-of-moments skew-normal (Robin's _fit; same as _fit_skewnorm)."""
    m, s = V.mean(axis=1), V.std(axis=1)
    sk = np.mean((V - m[:, None]) ** 3, axis=1) / s ** 3
    sp = np.minimum(0.99, np.abs(sk))
    delta = np.sqrt(np.pi / 2 * sp ** (2 / 3) / (sp ** (2 / 3) + ((4 - np.pi) / 2) ** (2 / 3))) * np.sign(sk)
    shape = delta / np.sqrt(1 - delta ** 2)
    scale = s / np.sqrt(1 - 2 * delta ** 2 / np.pi)
    return shape, m - scale * delta * np.sqrt(2 / np.pi), scale


def _ks_pass_rate(V):
    """Robin's _ks_test: fraction of resamples with sqrt(n)*max|F(v_(i)) - i/n| < 1.36."""
    shape, loc, scale = _fit_skewnorm_rs(V)
    n = V.shape[1]
    F = stats.skewnorm.cdf(np.sort(V, axis=1), shape[:, None], loc=loc[:, None], scale=scale[:, None])
    D = np.nanmax(np.abs(F - np.arange(1, n + 1) / n), axis=1)
    return float(np.nanmean(np.sqrt(n) * D < 1.36))


def _ci(v):
    return float(np.median(v)), float(np.quantile(v, 0.025)), float(np.quantile(v, 0.975))


def _sig(v):                                                       # Robin's _find_results: 95 % CI excludes 0
    lo, hi = np.quantile(v, [0.025, 0.975])
    return "*" if (lo > 0) or (hi < 0) else " "


def _partial_r(x, y, z):
    """Pearson partial correlation of x, y controlling for z (as in the cell-12 numbers)."""
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    r = np.corrcoef(rx, ry)[0, 1]
    dof = len(x) - 3
    return r, 2 * stats.t.sf(abs(r * np.sqrt(dof / (1 - r ** 2))), dof)


# ---- per-pool data (best N_CD, ranked by distance) ----
_POOL = {}
for label, group, _f in GROUPS:
    dates = case_dates_full[group][:N_CD]
    v = dict(y=_raw_anomaly(dates), rmst=_rmst_lookup.reindex(dates).to_numpy(),
             ridge=_rep_ridge.reindex(dates).to_numpy(),
             d4=d4_series.reindex(dates).to_numpy() / D4_NORM,
             old=target_region_anomaly(dates), rank=np.arange(len(dates)))
    ok = np.all([np.isfinite(a) for a in v.values()], axis=0)
    _POOL[group] = dict({k: a[ok] for k, a in v.items()}, label=label)

# ============================ [A] Robin attribution ============================
VARIANTS = [("RMST (Robin)", []), ("RMST + ridge", ["ridge"]),
            ("RMST + COL act.", ["d4"]), ("RMST + ridge + COL act.", ["ridge", "d4"])]
ATTR, _rows_a = {}, []
_rng_attr = np.random.default_rng(0)
print(f"[A] Robin attribution | RMST {ATTR_YEAR_PAST} = {RMST_PAST:.3f}, {ATTR_YEAR_PRESENT} = {RMST_PRESENT:.3f}, "
      f"dRMST = {DRMST:.3f} degC | B = {B_BOOT}, N = {N_CD} per pool | * = 95 % CI excludes 0")
print(f"    old projection: all-AMJJA-days slope = {RMST_ANOMALY_SLOPE:+.2f} degC/degC "
      f"-> change {RMST_ANOMALY_SLOPE * DRMST:+.2f} degC")
for label, group, _f in GROUPS:
    P = _POOL[group]
    idx = _rng_attr.choice(np.arange(P["y"].size), size=(B_BOOT, P["y"].size), replace=True)  # same resamples for all variants
    ATTR[group] = {}
    print(f"\n  {label} (n = {P['y'].size})")
    print(f"    {'covariates':24s} {'beta_RMST [degC/degC]':>24s}   {f'change {ATTR_YEAR_PAST}->{ATTR_YEAR_PRESENT} [degC]':>26s}   KS pass   extra coefs (median)")
    for vname, extra in VARIANTS:
        Xs = np.stack([P["rmst"][idx]] + [P[c][idx] for c in extra], axis=2)
        coef, R = _ols_rs(Xs, P["y"][idx])
        beta = coef[:, 1]; change = beta * DRMST; ks = _ks_pass_rate(R)
        ATTR[group][vname] = dict(beta=beta, change=change, ks=ks)
        bm, bl, bh = _ci(beta); cm, cl, ch = _ci(change)
        ex = "  ".join(f"{c}={np.median(coef[:, 2 + j]):+.4f}{_sig(coef[:, 2 + j])}" for j, c in enumerate(extra))
        print(f"    {vname:24s} {bm:+6.2f} [{bl:+6.2f}, {bh:+6.2f}]{_sig(beta)}   "
              f"{cm:+6.2f} [{cl:+6.2f}, {ch:+6.2f}]{_sig(change)}      {ks:.3f}   {ex}")
        _rows_a.append(dict(pool=label, covariates=vname, beta_med=bm, beta_lo=bl, beta_hi=bh,
                            change_med=cm, change_lo=cl, change_hi=ch, ks_pass_rate=ks))
print(f"\n  difference cut-off - no cut-off (change {ATTR_YEAR_PAST}->{ATTR_YEAR_PRESENT}):")
for vname, _e in VARIANTS:
    d = ATTR["COL_TROUGH"][vname]["change"] - ATTR["NO_COL_TROUGH"][vname]["change"]
    dm, dl, dh = _ci(d)
    print(f"    {vname:24s} {dm:+6.2f} [{dl:+6.2f}, {dh:+6.2f}]{_sig(d)}")
    _rows_a.append(dict(pool="cut-off minus no cut-off", covariates=vname, change_med=dm, change_lo=dl, change_hi=dh))
pd.DataFrame(_rows_a).to_csv("figure5b_robin_attribution.csv", index=False)

# ===================== [B] COL activity vs T2M, both frameworks =====================
FRAMEWORKS = [("old projection (all-AMJJA slope)", lambda P, g: P["old"]),
              ("Robin projection (pool's own slope)",
               lambda P, g: P["y"] - np.median(ATTR[g]["RMST (Robin)"]["beta"]) * (P["rmst"] - RMST_PRESENT))]
_rows_b = []
print(f"\n[B] COL activity vs present-climate T2M of the best-N analogues: r = Pearson, rho = Spearman")
for fname, fy in FRAMEWORKS:
    print(f"\n  ===== {fname} =====")
    for label, group, _f in GROUPS:
        P = _POOL[group]; t2 = fy(P, group)
        s_ridge = float(np.polyfit(P["ridge"], t2, 1)[0])                    # pool's ridge->T2M slope (N_CD)
        t2rr = t2 - s_ridge * (P["ridge"] - P["ridge"].mean())
        print(f"  {label}: ridge->T2M slope = {s_ridge:+.4f} degC/m")
        print(f"    {'N':>3s}  {'raw r (p)':>15s} {'raw rho (p)':>15s} | {'ridge-rem. r (p)':>16s} {'rho (p)':>15s} | {'partial r (p)':>15s}")
        for n in N_LIST_REPORT:
            m = P["rank"] < n
            x = P["d4"][m]
            r0, p0 = stats.pearsonr(x, t2[m]); s0, q0 = stats.spearmanr(x, t2[m])
            r1, p1 = stats.pearsonr(x, t2rr[m]); s1, q1 = stats.spearmanr(x, t2rr[m])
            r2, p2 = _partial_r(x, t2[m], P["ridge"][m])
            print(f"    {n:3d}  {r0:+.2f} ({p0:.3f})   {s0:+.2f} ({q0:.3f})  |  {r1:+.2f} ({p1:.3f})    {s1:+.2f} ({q1:.3f})  |  {r2:+.2f} ({p2:.3f})")
            _rows_b.append(dict(framework=fname, pool=label, N=n, ridge_slope=s_ridge,
                                r_raw=r0, p_raw=p0, rho_raw=s0, prho_raw=q0,
                                r_ridge=r1, p_ridge=p1, rho_ridge=s1, prho_ridge=q1,
                                r_partial=r2, p_partial=p2))
pd.DataFrame(_rows_b).to_csv("figure5b_correlations_ridge.csv", index=False)

### Numbers behind Figure 5b — cut-off sub-pool

In [ ]:
# ============================================================================
# Numbers behind Figure 5b — CUT-OFF sub-pool only (COL_TROUGH).
# Self-contained: builds the ridge index (detrended target-box Z500) internally.
# Needs cell-9 globals: fields, d4_series, D4_NORM, case_dates_full,
#                       target_region_anomaly, select_latitude_range.
# ============================================================================
from scipy import stats
rng = np.random.default_rng(0)
N = 85
GROUP = "COL_TROUGH"

# ---- ridge index = comp_target_box_mean of RMST-detrended Z500 over the TARGET box ----
_ridge_series = pd.Series(np.asarray(comp_target_box_mean(fields, "lon", "lat").values, float),
                          index=pd.DatetimeIndex(fields["time"].values).normalize())
def ridge_index(dates):
    idx = pd.DatetimeIndex(pd.to_datetime(np.asarray(dates))).normalize()
    return _ridge_series.reindex(idx).to_numpy()

# ---- Z500 SEASONAL ANOMALY: remove the fixed 1995-2025 +/-3 d doy climatology from the
#      (RMST-detrended) ridge-box Z500, EXACTLY as target_region_anomaly does for T2m.
#      The analogue dates are scattered across the season, so the ridge REMOVAL (not the
#      selection) must use the Z500 anomaly, else the seasonal cycle leaks into the residual.
_ridge_da = xr.DataArray(_ridge_series.values, dims="time",
                         coords={"time": _ridge_series.index.values})
_ridge_clim_doy = fixed_doy_climatology(_ridge_da, CLIM_START_YEAR, CLIM_END_YEAR, CLIM_HALF_WINDOW_DAYS)
_ridge_clim_lookup = pd.Series(np.asarray(_ridge_clim_doy.values, float),
                               index=np.asarray(_ridge_clim_doy["dayofyear"].values, int))
def ridge_index_anom(dates):
    idx = pd.DatetimeIndex(pd.to_datetime(np.asarray(dates))).normalize()
    return ridge_index(idx) - _ridge_clim_lookup.reindex(idx.dayofyear).to_numpy()

# ---- ridge-ANOMALY -> T2M regression, LEARNED over ALL MJJA days 1940-2025 (not the cut-off sub-pool) ----
# Uniform climatological ridge control on the Z500 ANOMALY (MJJA anomaly link; matches laptop Fig 2 z500anom).
_fitd = pd.DatetimeIndex(_ridge_series.dropna().index)
_fitd = _fitd[(_fitd.month >= 5) & (_fitd.month <= 8) & (_fitd.year <= LAST_COMPLETE_ERA5_YEAR)]
_zp, _tp = ridge_index_anom(_fitd), target_region_anomaly(_fitd)
_okz = np.isfinite(_zp) & np.isfinite(_tp)
RIDGE_SLOPE = float(np.polyfit(_zp[_okz], _tp[_okz], 1)[0])
RIDGE_REF   = 0.0     # Z500 anomaly already centred on the present-climate seasonal cycle
def ridge_removed_t2m(dates):
    return target_region_anomaly(dates) - RIDGE_SLOPE * (ridge_index_anom(dates) - RIDGE_REF)
print(f"ridge (target-box Z500 ANOMALY) -> T2M slope = {RIDGE_SLOPE:+.4f} K/m (MJJA anomaly link)\n")

# ---------------------------------------------------------------------------
def _pool(Ni):
    """best-Ni CUT-OFF analogues: normalized d4, raw T2M, ridge, ridge-removed T2M."""
    dts   = case_dates_full[GROUP][:Ni]
    d4    = d4_series.reindex(dts).to_numpy() / D4_NORM
    t2raw = target_region_anomaly(dts)
    zbox  = ridge_index_anom(dts)   # control for the Z500 ANOMALY (seasonal cycle removed)
    t2rr  = ridge_removed_t2m(dts)
    ok = np.isfinite(d4) & np.isfinite(t2raw) & np.isfinite(zbox) & np.isfinite(t2rr)
    return d4[ok], t2raw[ok], zbox[ok], t2rr[ok]

def _partial_r(x, y, z):
    """Pearson partial correlation of x,y controlling for z (residualize both)."""
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    r  = np.corrcoef(rx, ry)[0, 1]
    dof = len(x) - 3
    t  = r * np.sqrt(dof / (1 - r**2))
    return r, 2 * stats.t.sf(abs(t), dof)

d4, t2raw, zbox, t2rr = _pool(N)
n = len(d4)
print(f"cut-off sub-pool = best-{N} COL_TROUGH analogues: n = {n} days")
print(f"climatological cut-off intensity  D4_NORM (So) = {D4_NORM:.1f} m (100 km)^-1\n")

# [1] raw link within the cut-off sub-pool
r_raw, p_raw = stats.pearsonr(d4, t2raw); rho_raw, _ = stats.spearmanr(d4, t2raw)
print("[1] raw COL activity vs present-climate T2M:")
print(f"    r = {r_raw:.2f}  ->  r^2 = {r_raw**2*100:.0f}% of variance   (p = {p_raw:.2g}, Spearman = {rho_raw:.2f})\n")

# [2] ridge-controlled framework
r_rr, p_rr = stats.pearsonr(d4, t2rr); rho_rr, p_rho = stats.spearmanr(d4, t2rr)
r_par, p_par = _partial_r(d4, t2raw, zbox)
print("[2] ridge-controlled (residual from Z500 box-mean):")
print(f"    partial r = {r_par:.2f} (p = {p_par:.3f})  ->  {r_par**2*100:.0f}% of variance beyond Z500 covariance")
print(f"    d4 vs ridge-removed T2M: r = {r_rr:.2f} (p = {p_rr:.3f}), Spearman = {rho_rr:.2f} (p = {p_rho:.3f})\n")

# [3] decline of ridge-controlled variance with pool size
print("[3] ridge-controlled variance vs pool size:")
for Ni in [20,25, 30, 40, 50, 60, 70, 85]:
    xi, _, zi, yi = _pool(Ni)
    ri, pi = _partial_r(xi, yi, zi)
    print(f"    N={Ni:>2}: partial r = {ri:+.2f}, r^2 = {ri**2*100:4.0f}%   (p = {pi:.3g}){'  *sig' if pi < 0.05 else ''}")
print()

# [4] slope + bootstrap CI at N=85
slope = np.polyfit(d4, t2rr, 1)[0]
mean_d4 = float(np.nanmean(d4))
boot = np.array([np.polyfit(d4[s], t2rr[s], 1)[0]
                 for s in (rng.integers(0, n, n) for _ in range(5000))])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"[4] ridge-controlled slope (per +1 day of average cut-off intensity, So = {D4_NORM:.0f} m/100km):")
print(f"    dT2M/dd4 = {slope:.2f} K  [{lo:.2f}, {hi:.2f}]")
print(f"    mean(d4 | cut-off sub-pool) = {mean_d4:.1f} days")
print(f"    -> residual T2M anomaly at controlled Z500 = "
      f"{slope*mean_d4:.1f} K  [{lo*mean_d4:.1f}, {hi*mean_d4:.1f}]")

In [ ]:
doy = EVENT.dayofyear                                   # 25 May 2026 -> 145
clim_val = float(target_clim_doy.sel(dayofyear=doy))
obs_val  = float(target_box_series.sel(time=np.datetime64(EVENT.to_datetime64())))
print(f"Target-box climatology {EVENT:%d %b} (doy {doy}, "
      f"{CLIM_START_YEAR}-{CLIM_END_YEAR}, ±{CLIM_HALF_WINDOW_DAYS} d): {clim_val:.2f} °C")
print(f"Observed {EVENT:%Y-%m-%d}: {obs_val:.2f} °C  ->  raw anomaly {obs_val - clim_val:+.2f} °C")
print(f"Present-climate anomaly (used in Fig. 5b): {event_target_anomaly:+.2f} °C")

### Numbers for the Fig. 5 results text — COL activity link, cut-off vs no-cut-off, 25 May 2026 vs analogues

In [ ]:
# ============================================================================
# Numbers quoted in the Fig. 5 results text, paragraph by paragraph.
#   [1] COL activity vs present-day T2M anomaly (cut-off pool; no-cut-off pool for contrast)
#   [2] binary distinction: cut-off vs no-cut-off present-day T2M, raw and ridge-controlled
#   [3] 25 May 2026 vs the analogues: T2M anomaly, preceding COL activity, ridge strength
# Ridge = comp_target_box_mean of the RMST-detrended Z500 over the TARGET box (as in the cell above).
# ============================================================================
import statsmodels.api as sm
N_SWEEP = [20, 25, 30, 40, 50, 60, 70, 85]
_C, _NC = case_dates_full["COL_TROUGH"], case_dates_full["NO_COL_TROUGH"]
_tz_box = comp_target_box_mean(fields, "lon", "lat").load()
_txt_ridge = pd.Series(np.asarray(_tz_box.values, float), index=pd.DatetimeIndex(_tz_box["time"].values).normalize())
# Z500 SEASONAL ANOMALY for the ridge control in [1]/[2]: remove the 1995-2025 +/-3 d doy climatology
# (analogue dates scatter across the season, so the ridge removal must use the anomaly, not absolute Z500).
_txt_rc = fixed_doy_climatology(_tz_box, CLIM_START_YEAR, CLIM_END_YEAR, CLIM_HALF_WINDOW_DAYS)
_txt_ridge_anom = _txt_ridge - _txt_rc.sel(dayofyear=xr.DataArray(_txt_ridge.index.dayofyear.values, dims="t")).values
_dist = pd.Series(dist_to_event, index=pool_times)


def _cols(dates):
    """COL activity (x clim), present-day box T2M anomaly, ridge (m) for the given analogue dates."""
    dates = pd.DatetimeIndex(dates)
    x = d4_series.reindex(dates).to_numpy() / D4_NORM
    y = target_region_anomaly(dates)
    z = _txt_ridge_anom.reindex(dates).to_numpy()   # Z500 anomaly (seasonal cycle removed)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    return x[ok], y[ok], z[ok]


def _txt_partial_r(x, y, z):
    """Pearson partial correlation of x, y controlling for z (residualise both on z)."""
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    r = np.corrcoef(rx, ry)[0, 1]
    dof = len(x) - 3
    return r, 2 * stats.t.sf(abs(r * np.sqrt(dof / (1 - r ** 2))), dof)


# ============================ [1] COL activity -> T2M ============================
x, y, z = _cols(_C[:N_CD])
r, p = stats.pearsonr(x, y); rho, prho = stats.spearmanr(x, y)
yrr = y - np.polyfit(z, y, 1)[0] * (z - z.mean())                 # ridge-removed T2M
rpar, ppar = _txt_partial_r(x, y, z)
rr, prr = stats.pearsonr(x, yrr); rhor, prhor = stats.spearmanr(x, yrr)
print(f"[1] COL activity vs present-day T2M anomaly -- cut-off pool, N = {N_CD}")
print(f"    raw:              r = {r:.2f} (r^2 = {r * r * 100:.0f} %, p = {p:.4f}), Spearman = {rho:.2f} (p = {prho:.3f})")
print(f"    ridge-controlled: partial r = {rpar:.2f} (r^2 = {rpar ** 2 * 100:.0f} %, p = {ppar:.4f}); "
      f"ridge-removed T2M r = {rr:.2f} (p = {prr:.4f}), Spearman = {rhor:.2f} (p = {prhor:.3f})")
print(f"    pool-size dependence (best N):")
print(f"      {'N':>3s} | cut-off: raw r (p)   partial r, r^2 (p)    | no cut-off: raw r (p)   partial r (p)")
for n in N_SWEEP:
    xc, yc, zc = _cols(_C[:n]); xn, yn, zn = _cols(_NC[:n])
    rc, pc = stats.pearsonr(xc, yc); prc, ppc = _txt_partial_r(xc, yc, zc)
    rn, pn = stats.pearsonr(xn, yn); prn, ppn = _txt_partial_r(xn, yn, zn)
    print(f"      {n:3d} |   {rc:+.2f} ({pc:.3f})    {prc:+.2f}, {prc ** 2 * 100:3.0f} % ({ppc:.3f}){'*' if ppc < 0.05 else ' '}  |"
          f"    {rn:+.2f} ({pn:.3f})       {prn:+.2f} ({ppn:.3f})")
_rng_txt = np.random.default_rng(0)                               # same bootstrap as the cell above
slope = np.polyfit(x, yrr, 1)[0]
boot = np.array([np.polyfit(x[s], yrr[s], 1)[0] for s in (_rng_txt.integers(0, x.size, x.size) for _ in range(5000))])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"    ridge-controlled slope = {slope:.2f} K [{lo:.2f}, {hi:.2f}] per 1 day of climatological-mean cut-off "
      f"intensity (D4_NORM = So {D4_NORM:.1f} m (100 km)^-1)")
print(f"    mean COL activity of the cut-off pool = {x.mean():.2f} -> residual T2M anomaly at controlled Z500 = "
      f"{slope * x.mean():.2f} K [{lo * x.mean():.2f}, {hi * x.mean():.2f}]")

# ===================== [2] binary distinction cut-off vs no cut-off =====================
print(f"\n[2] cut-off minus no-cut-off present-day T2M anomaly (Welch t-test 95 % CI; two-sample K-S)")
print(f"    ridge-controlled = residual after the pooled ridge->T2M regression; ANCOVA = cut-off term in T2M ~ ridge + cut-off")
print(f"    {'N':>3s}  {'diff [95 % CI] K':>22s}  Welch p   K-S p | ridge-ctrl diff (p) | ANCOVA (p)    | "
      f"ridge cut-no [m] (p) | mean Z500 dist cut / no")
for n in N_SWEEP:
    _, a, za = _cols(_C[:n]); _, b, zb = _cols(_NC[:n])
    w = stats.ttest_ind(a, b, equal_var=False); ci = w.confidence_interval(0.95); ks = stats.ks_2samp(a, b)
    zz, yy = np.r_[za, zb], np.r_[a, b]
    s_z = np.polyfit(zz, yy, 1)[0]
    ra, rb = a - s_z * (za - zz.mean()), b - s_z * (zb - zz.mean())
    w2 = stats.ttest_ind(ra, rb, equal_var=False)
    anc = sm.OLS(yy, np.c_[np.ones(yy.size), zz, np.r_[np.ones(a.size), np.zeros(b.size)]]).fit()
    wz = stats.ttest_ind(za, zb, equal_var=False)
    da, db = _dist.reindex(_C[:n]).mean(), _dist.reindex(_NC[:n]).mean()
    print(f"    {n:3d}  {a.mean() - b.mean():+.2f} [{ci.low:+.2f}, {ci.high:+.2f}]    {w.pvalue:.3f}   {ks.pvalue:.3f} |"
          f"   {ra.mean() - rb.mean():+.2f} ({w2.pvalue:.3f})    | {anc.params[2]:+.2f} ({anc.pvalues[2]:.3f}) |"
          f"   {za.mean() - zb.mean():+5.1f} ({wz.pvalue:.3f})    |  {da:.0f} / {db:.0f}")

# ===================== [3] 25 May 2026 vs the analogues =====================
_dd = pd.DatetimeIndex(np.r_[_C[:N_CD], _NC[:N_CD]]); _aa = target_region_anomaly(_dd)
print(f"\n[3] 25 May 2026 vs the {_dd.size} analogues (best {N_CD} per pool)")
print(f"    present-day T2M anomaly: event {event_target_anomaly:+.2f} degC | warmest analogue "
      f"{np.nanmax(_aa):+.2f} degC on {_dd[np.nanargmax(_aa)].date()} | event above all analogues: "
      f"{event_target_anomaly > np.nanmax(_aa)}")
for _d in _dd[_dd.year == 2022]:
    print(f"      2022 analogue {_d.date()} ({'cut-off' if _d in set(_C[:N_CD]) else 'no cut-off'}): "
          f"{float(target_region_anomaly([_d])[0]):+.2f} degC")
_ev_d4 = D4_2026[str(EVENT.date())] / D4_NORM
_xc = d4_series.reindex(_C[:N_CD]).to_numpy() / D4_NORM
print(f"    preceding COL activity: event {_ev_d4:.2f} x clim | cut-off analogues median {np.median(_xc):.2f}, "
      f"event rank {int((_xc > _ev_d4).sum()) + 1}/{_xc.size} ({stats.percentileofscore(_xc, _ev_d4):.0f}th percentile)")
_rc = _txt_rc                       # reuse the climatology built above (same 1995-2025 +/-3 d)
_ra = _txt_ridge_anom              # Z500 anomaly series built above
_ev_ridge = float(comp_target_box_mean(event_field, "lon", "lat"))
_ev_ra = _ev_ridge - float(_rc.sel(dayofyear=EVENT.dayofyear))
_may = _ra[_ra.index.month == EVENT.month]; _an = _ra.reindex(_dd)
print(f"    ridge (target-box detrended Z500): event {_ev_ridge:.1f} m = {_ev_ra:+.1f} m vs the "
      f"{CLIM_START_YEAR}-{CLIM_END_YEAR} calendar-day climatology")
print(f"      rank among May days 1940-{LAST_COMPLETE_ERA5_YEAR}: {int((_may.values > _ev_ra).sum()) + 1}/{_may.size} "
      f"(runner-up {_may.max():+.1f} m on {_may.idxmax().date()})")
print(f"      rank among the {_dd.size} analogues: {int((_an.values > _ev_ra).sum()) + 1}/{_dd.size} "
      f"(runner-up {_an.max():+.1f} m on {_an.idxmax().date()})")
print(f"      rank among all AMJJA days 1940-{LAST_COMPLETE_ERA5_YEAR}: {int((_ra.values > _ev_ra).sum()) + 1}/{_ra.size} "
      f"(max {_ra.max():+.1f} m on {_ra.idxmax().date()})")

### Best 30 cut-off analogues (date list)

In [ ]:
# ---- best-N cut-off analogues of the 25 May 2026 event, best first ----
# (RMST-detrended Z500 distance in the analogue box, +/-5-day independence, >=1 closed cut-off with So>=6 in the large box)
N_PRINT = 30
print("# --- ACTIVE: factual AMJJA (only cut-off) ---")
for _d in case_dates_full["COL_TROUGH"][:N_PRINT]:
    print(_d.strftime("%Y-%m-%d"))

### Supplement — best 2022 analogue & 25 June 2026 heatwave

In [ ]:
# ============================================================================
# SUPPLEMENT: COL activity vs present-climate T2M — two reference patterns,
# each projected to present-day climate INDEPENDENTLY (per-case RMST anchor).
#   left  = best 2022 analogue of the 25 May event   |   right = 25 June 2026 heatwave
# ============================================================================
from matplotlib.ticker import FuncFormatter

# ---- font scheme, identical to Figs 2–5 (single base fs, offsets not hardcoded) ----
fs    = 14
axfs  = fs-2   # x/y tick label size
legfs = fs-4   # in-panel legends / stats boxes
plt.rcParams.update({"font.size":fs,"axes.titlesize":fs+1,"axes.labelsize":fs,
                     "xtick.labelsize":axfs,"ytick.labelsize":axfs})
FS_L   = fs      # axis labels
FS_T   = axfs    # small marginal-axis labels ("count", "PDF")
FS_LEG = legfs   # in-panel legends

def _letter(ax, s, x=-0.12, dy=6):   # dy in points: same gap above the top edge for every panel
    ax.annotate(s, xy=(x, 1.0), xycoords="axes fraction",
                xytext=(0, dy), textcoords="offset points",
                fontweight="bold", fontsize=fs+1, va="bottom", ha="right")

N = 85

# ---- reference patterns ----
_in2022 = np.where(pool_times.year == 2022)[0]
REF_2022 = pool_times[_in2022[np.argmin(dist_to_event[_in2022])]]     # closest 2022 analogue
JUNE_HW  = pd.Timestamp("2026-06-25")

# ---- per-case present-climate projection (anchor = ref_date's own present-day RMST) ----
def _present_anomaly(dates, ref_date):
    idx = pd.DatetimeIndex(pd.to_datetime(np.asarray(dates))).normalize()
    anom = _raw_anomaly(idx)
    if PRESENT_CLIMATE_DETREND:
        ry  = LAST_COMPLETE_ERA5_YEAR                                # present-day anchor for every case
        # ry = min(pd.Timestamp(ref_date).year, LAST_COMPLETE_ERA5_YEAR)  # <- use each case's OWN year instead
        rts = pd.Timestamp(year=ry, month=pd.Timestamp(ref_date).month,
                           day=pd.Timestamp(ref_date).day).normalize()
        rmst_ref = float(RMST.sel(time=np.datetime64(rts), method="nearest"))
        anom = anom + RMST_ANOMALY_SLOPE * (rmst_ref - _rmst_lookup.reindex(idx).values)
    return anom

def _rank_to(ref_date):
    diff = pool_values - _detrended_z500_field(ref_date)
    dist = np.sqrt((diff * diff).reshape(pool_values.shape[0], -1).sum(axis=1))
    r = greedy_independent(dist, pool_times)
    return {"COL_TROUGH":    pool_times[r[pool_has_feature[r]][:N_SELECT]],
            "NO_COL_TROUGH": pool_times[r[~pool_has_feature[r]][:N_SELECT]]}

def _ref_d4(ref_date):
    v = float(d4_series.get(pd.Timestamp(ref_date).normalize(), np.nan))
    if not np.isfinite(v):
        v = D4_2026.get(str(pd.Timestamp(ref_date).date())) if "D4_2026" in globals() else None
        if v is None and "detect_d4" in globals():
            v = detect_d4(pd.Timestamp(ref_date))
    return (v / D4_NORM) if v is not None and np.isfinite(v) else None

def _draw_panel_b(ax_s, ax_m, dates_by_group, ref_date):
    anoms = {}
    for label, group, _f in GROUPS:
        dates = dates_by_group[group][:N]
        d4_all = d4_series.reindex(dates).to_numpy() / D4_NORM
        t2_all = _present_anomaly(dates, ref_date)                   # <- per-case projection
        rank = np.arange(len(dates)); ok = np.isfinite(d4_all) & np.isfinite(t2_all)
        d4, t2, rank = d4_all[ok], t2_all[ok], rank[ok]
        top = rank < NTOP; col = case_colors[group]; anoms[group] = t2
        ax_s.scatter(d4[~top], t2[~top], s=14, color=col, alpha=0.22, edgecolors="none")
        ax_s.scatter(d4[top],  t2[top],  s=34, color=col, alpha=0.9, edgecolors="k", linewidths=0.5)
        f = _fit_stats(d4, t2)
        if f:
            rp, pp, rs, ps, b, a = f; xs = np.linspace(d4.min(), d4.max(), 100)
            ax_s.plot(xs, a + b * xs, color=col, lw=2.0, label=f"{label} N={N}: r={rp:+.2f} (p={pp:.2g})")
    xr = _ref_d4(ref_date); yr = float(_present_anomaly([ref_date], ref_date)[0])
    if xr is not None:
        ax_s.scatter(xr, yr, s=90, color="black", marker="*", zorder=9)
    ax_s.set_xlabel(r"5 Day COL Activity [Clim. Mean]", fontsize=FS_L)
    ax_s.set_ylabel("UK-France T2M anomaly, present [°C]", fontsize=FS_L)
    ax_s.grid(alpha=0.3); ax_s.legend(fontsize=FS_LEG, frameon=True, loc="lower right")
    _all = np.concatenate([anoms[g] for _, g, _ in GROUPS])
    ygrid = np.linspace(_all.min() - 1.0, _all.max() + 1.0, 300)
    for label, group, _f in GROUPS:
        col = case_colors[group]; ap = anoms[group]
        ax_m.hist(ap, bins=18, density=True, orientation="horizontal", histtype="stepfilled", alpha=0.18, color=col)
        sh, lo, sc = _fit_skewnorm(ap)
        ax_m.plot(stats.skewnorm.pdf(ygrid, sh, loc=lo, scale=sc), ygrid, color=col, lw=1.6)
        ax_m.axhline(float(np.mean(ap)), color=col, ls="--", lw=0.9)
    ax_m.axhline(yr, color="black", lw=1.1)
    for thr in ANOMALY_THRESHOLDS_C:
        ax_m.axhline(thr, color="grey", ls=":", lw=0.6)
    ax_m.tick_params(labelleft=False, labelsize=FS_T); ax_m.set_xlabel("PDF", fontsize=FS_T)
    ax_m.xaxis.set_major_formatter(FuncFormatter(lambda v, _p: "" if abs(v) < 1e-9 else f"{v:g}"))
    ax_m.grid(linestyle="dotted", linewidth=0.3)

# ---- 2-column figure (same row height & panel-b subgrid proportions as Fig 5) ----
fig = plt.figure(figsize=(14, 4))
outer = fig.add_gridspec(1, 2, wspace=0.22)
panels = [(REF_2022, f"July 2022 Heatwave ({REF_2022.strftime('%d %b')})", "(a)"),
          (JUNE_HW,  "June 2026 heatwave (25 Jun)", "(b)")]
_ax0 = None
for k, (ref, title, lett) in enumerate(panels):
    gs = outer[0, k].subgridspec(1, 2, width_ratios=[4, 1.25], wspace=0.03)
    ax_s = fig.add_subplot(gs[0, 0], sharey=_ax0); _ax0 = _ax0 or ax_s
    ax_m = fig.add_subplot(gs[0, 1], sharey=ax_s)
    _draw_panel_b(ax_s, ax_m, _rank_to(ref), ref)
    ax_s.set_title(title, fontsize=fs); _letter(ax_s, lett, x=-0.2)

print(f"best 2022 analogue = {REF_2022.date()} (dist-to-event = {dist_to_event[_in2022].min():.2f})")
fig.savefig("figureS_2022_june2026_clim1995-2025.pdf", bbox_inches="tight")
plt.show()